In [1]:
import pandas as pd
import pickle as pkl
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import GridSearchCV
from sklearn import metrics
from sklearn.pipeline import Pipeline

In [2]:
neg_to_pos_ratio = 1
seed = 42

In [3]:
filename_pos_human = 'pubmed_articles_pos_human.csv'

pos_data_human = pd.read_csv(
         filename_pos_human,
)


In [4]:
filename_neg_human = 'pubmed_articles_neg_human.csv'

neg_data_human = pd.read_csv(filename_neg_human)

In [5]:
with open('pos_neg_dataset_ids.pkl', 'rb') as f:
    prop_typ_id_dict= pkl.load(f)

In [6]:
prop_typ_id_dict.keys()

dict_keys(['is_rct', 'is_sr', 'is_casecontrol', 'is_ct', 'is_cohort', 'is_guideline', 'is_consensus', 'is_qual', 'is_ctprotocol'])

In [7]:
sr_pos_ids = prop_typ_id_dict['is_ct']['pos']
sr_pos_set  = pos_data_human.loc[sr_pos_ids]
sr_pos_set['is_ct'] = int(1)

In [8]:
sr_neg_ids = prop_typ_id_dict['is_ct']['neg']
sr_neg_set  = neg_data_human.loc[sr_neg_ids]
sr_neg_set['is_ct'] = int(0)

In [9]:
sr_neg_set_sample = sr_neg_set.sample(n=len(sr_pos_set)*neg_to_pos_ratio)

In [10]:
len(sr_neg_set_sample)

593277

In [11]:
sr_data = pd.concat([sr_neg_set_sample, sr_pos_set]).sample(frac=1)

In [12]:
sr_data.columns

Index(['Unnamed: 0.1', 'index', 'Unnamed: 0', 'pmid', 'status',
       'indexing_method', 'title', 'abstract', 'abstract_plaintext', 'authors',
       'journal', 'journal_abbrv', 'year', 'mesh', 'month', 'volume', 'issue',
       'pages', 'ptyp', 'registry_ids', 'dois', 'is_ct', 'unnamed: 0'],
      dtype='object')

In [13]:
sr_data = sr_data.drop(columns = ['Unnamed: 0', 'unnamed: 0', 'indexing_method', 'abstract', 'journal_abbrv', 'year', 'mesh', 'month', 'volume', 'issue', 'pages',
       'ptyp', 'registry_ids', 'dois',])

In [14]:
sr_labels = sr_data['is_ct']
sr_inputs = sr_data['abstract_plaintext']
# sr_inputs = sr_data['title'] + sr_data['abstract_plaintext']

In [15]:
sr_inputs

2724784    We have performed arthroscopic Bankart procedu...
2621155    The subjects were 36 hypertensive patients age...
4796769    BACKGROUND\n\n\nOlder patients with advanced b...
8155224    To test the effectiveness of betahistine HC1 i...
5027421    Background\n\n\nCervicobrachial pain (CP) is a...
                                 ...                        
3801056    Information-intensive Web services such as pri...
3760503    This study compared DermaMend and A & D for tr...
6394152    Fifty-six and forty patients with acute icteri...
5145389    A multicenter study was undertaken to determin...
3893529    Twenty-one pretreatment variables were assesse...
Name: abstract_plaintext, Length: 1186554, dtype: object

In [16]:
X_train, X_test_val, y_train, y_test_val =  train_test_split(sr_inputs, sr_labels, test_size=0.2)

In [17]:
X_val, X_test, y_val, y_test =  train_test_split(X_test_val, y_test_val, test_size=0.5)

In [5]:
import os
os.path.join('model','pubmed_is_ct_trained_lr.pkl')

'model/pubmed_is_ct_trained_lr.pkl'

In [9]:
# with open(os.path.join('model','pubmed_is_ct_trained_lr.pkl'), 'wb') as f:
with open('pubmed_is_ct_trained_lr.pkl', 'wb') as f:
     print('opened')

opened


In [18]:
vec = HashingVectorizer(ngram_range=(1, 4), stop_words='english')

In [19]:
X_train_trans = vec.fit_transform(X_train.values.astype('U'))
# X_crowd = vec.transform(crowd['ti'] + '\n\n' + crowd['ab'])

In [20]:
grid = {"loss": ["log"],
        "penalty": ["l2"],
        "alpha": np.logspace(-10, -5, 20)}

In [21]:
# clf = Pipeline([('HashingVectorizer',vec),('GridSearchCVLR',GridSearchCV(SGDClassifier(), grid, n_jobs=4, scoring='roc_auc',verbose=100))])

In [22]:
clf = GridSearchCV(SGDClassifier(), grid, n_jobs=4, scoring='roc_auc',verbose=100)

In [1]:
clf.fit(X_train.values.astype('U'), y_train)

NameError: name 'clf' is not defined

In [ ]:
clf.best_estimator_

In [ ]:
with open('model/pubmed_is_ct_trained_lr.pck', 'wb') as f:
    pkl.dump(clf.best_estimator_, f)

In [ ]:
X_test_trans = vec.transform(X_test.values.astype('U'))
preds = clf.predict_proba(X_test_trans)

In [ ]:
preds = clf.predict_proba(X_test.values.astype('U'))

In [ ]:
preds

In [ ]:
metrics.plot_roc_curve(clf, X_test_trans, y_test) 

In [ ]:
y_pred = clf.predict(X_test_trans)

In [ ]:
y_pred

In [ ]:
metrics.precision_score(y_pred, y_test)

In [ ]:
metrics.recall_score(y_pred, y_test)

In [ ]:
metrics.f1_score(y_pred, y_test)

In [ ]:
metrics.accuracy_score(y_pred, y_test)

In [ ]:
len(y_test)